<a href="https://colab.research.google.com/github/Gajalakshmi993/Time-Based-Browsing-Pattern-Analyzer/blob/main/Final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Preprocessing

In [ ]:
import pandas as pd
from urllib.parse import urlparse

def extract_domain(url):
    try:
        return urlparse(url).netloc
    except:
        return None

def preprocess_data(history_path, category_map_path):
    df = pd.read_csv(history_path)

    # Extract domain
    df['domain'] = df['url'].apply(extract_domain)

    # Convert timestamp
    df['timestamp'] = pd.to_datetime(df['timestamp'])

    # Time features
    df['hour'] = df['timestamp'].dt.hour
    df['day'] = df['timestamp'].dt.day_name()

    # Load category mapping
    cat_map = pd.read_csv(category_map_path)
    df = df.merge(cat_map, on='domain', how='left')

    # Fill unknown category
    df['category'] = df['category'].fillna('others')

    return df

##Sessionization

In [ ]:
import pandas as pd

def create_sessions(df, gap_minutes=15):
    df = df.sort_values('timestamp')

    df['time_diff'] = df['timestamp'].diff().dt.total_seconds() / 60
    df['new_session'] = df['time_diff'] > gap_minutes

    df['session_id'] = df['new_session'].cumsum()

    session_features = df.groupby('session_id').agg({
        'category': lambda x: list(x),
        'domain': 'nunique',
        'timestamp': ['min', 'max'],
        'hour': 'mean'
    })

    session_features.columns = ['categories', 'unique_domains', 'start_time', 'end_time', 'avg_hour']

    return df, session_features.reset_index()

##RAM Merge (time alignment)

In [ ]:
import pandas as pd

def merge_ram(history_df, ram_df):
    ram_df['timestamp'] = pd.to_datetime(ram_df['timestamp'])

    merged = pd.merge_asof(
        history_df.sort_values('timestamp'),
        ram_df.sort_values('timestamp'),
        on='timestamp',
        direction='nearest'
    )

    return merged

##clustering

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

def cluster_sessions(session_df, n_clusters=3):
    features = session_df[['unique_domains', 'avg_hour']]

    scaler = StandardScaler()
    X = scaler.fit_transform(features)

    model = KMeans(n_clusters=n_clusters, random_state=42)
    session_df['cluster'] = model.fit_predict(X)

    return session_df, model

##LSTM

In [ ]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.sequence import pad_sequences

def train_lstm(session_df):
    sequences = session_df['categories']

    # Flatten categories
    all_cats = [cat for seq in sequences for cat in seq]

    le = LabelEncoder()
    le.fit(all_cats)

    encoded = [le.transform(seq) for seq in sequences]

    X, y = [], []

    for seq in encoded:
        for i in range(1, len(seq)):
            X.append(seq[:i])
            y.append(seq[i])

    X = pad_sequences(X, maxlen=10)
    y = np.array(y)

    model = Sequential([
        Embedding(input_dim=len(le.classes_), output_dim=16, input_length=10),
        LSTM(64),
        Dense(len(le.classes_), activation='softmax')
    ])

    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    model.fit(X, y, epochs=5, batch_size=32)

    return model, le

##Autoencoder (Anomaly Detection)

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
import numpy as np

def train_autoencoder(X):
    input_dim = X.shape[1]

    input_layer = Input(shape=(input_dim,))
    encoded = Dense(16, activation='relu')(input_layer)
    decoded = Dense(input_dim, activation='linear')(encoded)

    autoencoder = Model(input_layer, decoded)
    autoencoder.compile(optimizer='adam', loss='mse')

    autoencoder.fit(X, X, epochs=20, batch_size=32)

    return autoencoder

def detect_anomalies(model, X):
    recon = model.predict(X)
    error = ((X - recon) ** 2).mean(axis=1)

    threshold = np.percentile(error, 95)
    return error > threshold

In [ ]:
def generate_recommendations(session_df):
    recommendations = []

    late_sessions = session_df[session_df['avg_hour'] > 22]
    if len(late_sessions) > 0:
        recommendations.append("Reduce late-night browsing for better sleep.")

    heavy_sessions = session_df[session_df['unique_domains'] > 10]
    if len(heavy_sessions) > 0:
        recommendations.append("Too many tabs opened — may increase RAM usage.")

    return recommendations

##Pipeline

In [ ]:
from src.preprocess import preprocess_data
from src.sessionize import create_sessions
from src.ram_merge import merge_ram
from src.clustering import cluster_sessions
from src.lstm_model import train_lstm
from src.recommend import generate_recommendations

# Load data
history_path = "data/browsing_history.csv"
ram_path = "data/ram_log.csv"
category_map = "data/domain_category_map.csv"

# Step 1: Preprocess
df = preprocess_data(history_path, category_map)

# Step 2: Sessionize
df, session_df = create_sessions(df)

# Step 3: RAM Merge
import pandas as pd
ram_df = pd.read_csv(ram_path)
df = merge_ram(df, ram_df)

# Step 4: Clustering
session_df, model = cluster_sessions(session_df)

# Step 5: LSTM
lstm_model, encoder = train_lstm(session_df)

# Step 6: Recommendations
recs = generate_recommendations(session_df)

print("Recommendations:")
for r in recs:
    print("-", r)

##Streamlit app

In [ ]:
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt

from src.preprocess import preprocess_data
from src.sessionize import create_sessions
from src.ram_merge import merge_ram
from src.clustering import cluster_sessions
from src.recommend import generate_recommendations

st.set_page_config(page_title="Browsing Pattern Analyzer", layout="wide")

st.title("🧠 Time-Based Browsing Pattern Analyzer")

# --------------------------
# Sidebar Inputs
# --------------------------
st.sidebar.header("⚙️ Settings")

history_file = st.sidebar.file_uploader("Upload Browsing History CSV")
ram_file = st.sidebar.file_uploader("Upload RAM Log CSV")
category_file = st.sidebar.file_uploader("Upload Domain Category Map CSV")

window_days = st.sidebar.selectbox("Select Time Window", [3, 4, 5])

# --------------------------
# Load Data
# --------------------------
if history_file and ram_file and category_file:

    df = preprocess_data(history_file, category_file)
    ram_df = pd.read_csv(ram_file)

    # Filter last N days
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    cutoff = df['timestamp'].max() - pd.Timedelta(days=window_days)
    df = df[df['timestamp'] >= cutoff]

    # Sessionization
    df, session_df = create_sessions(df)

    # RAM Merge
    df = merge_ram(df, ram_df)

    # Clustering
    session_df, model = cluster_sessions(session_df)

    # --------------------------
    # 1. Top Domains
    # --------------------------
    st.subheader("🌐 Top Domains")

    top_domains = df['domain'].value_counts().head(10)
    st.bar_chart(top_domains)

    # --------------------------
    # 2. Category Distribution
    # --------------------------
    st.subheader("📊 Category Distribution")

    cat_counts = df['category'].value_counts()
    st.bar_chart(cat_counts)

    # --------------------------
    # 3. Hourly Usage
    # --------------------------
    st.subheader("⏰ Hourly Usage Pattern")

    hourly = df.groupby('hour').size()

    fig, ax = plt.subplots()
    ax.plot(hourly.index, hourly.values)
    ax.set_xlabel("Hour")
    ax.set_ylabel("Visits")
    st.pyplot(fig)

    # --------------------------
    # 4. Session Clusters
    # --------------------------
    st.subheader("🧩 Session Clusters")

    st.dataframe(session_df[['session_id', 'cluster', 'unique_domains', 'avg_hour']])

    # --------------------------
    # 5. RAM Analysis
    # --------------------------
    st.subheader("💻 RAM Usage by Category")

    ram_analysis = df.groupby('category')['browser_ram_mb'].mean().sort_values(ascending=False)
    st.bar_chart(ram_analysis)

    # --------------------------
    # 6. Recommendations
    # --------------------------
    st.subheader("💡 Recommendations")

    recs = generate_recommendations(session_df)

    for r in recs:
        st.success(r)

else:
    st.info("👈 Upload all required files to start analysis")